In [3]:
# Import the libraries used to stress-test the classification result.

import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score


FEATURE_PATH = Path("door_cycle_features.csv")

print("Feature file exists:", FEATURE_PATH.exists())

cycle_features = pd.read_csv(FEATURE_PATH)

print("Feature table shape:", cycle_features.shape)
display(cycle_features.head())

Feature file exists: True
Feature table shape: (110, 143)


,Motor current(mA)_mean,Motor current(mA)_std,Motor current(mA)_min,Motor current(mA)_max,Motor current(mA)_range,Motor Voltage(10mV)_mean,Motor Voltage(10mV)_std,Motor Voltage(10mV)_min,Motor Voltage(10mV)_max,Motor Voltage(10mV)_range,...,position_zero_change_fraction,current_abs_integral,voltage_abs_integral,force_abs_integral,duration_seconds,n_rows,is_close,segment_id,operation,status
0,430.698925,523.670601,0.0,2208.0,2208.0,4146.236559,1416.459249,400.0,6200.0,5800.0,...,0.070270,80110.0,771200.0,165925.0,3.70,186,1,train_seg_001,Close,Normal
1,636.265734,727.547100,6.0,2493.0,2487.0,5521.678322,3184.691088,300.0,9100.0,8800.0,...,0.147887,90986.0,789600.0,168558.0,2.84,143,0,train_seg_002,Open,Normal
2,862.452555,777.944043,9.0,2493.0,2484.0,6197.080292,3478.273370,400.0,10400.0,10000.0,...,0.147059,118156.0,849000.0,163363.0,2.72,137,0,train_seg_003,Open,Abnormal resistance
3,576.577540,457.693062,112.0,1993.0,1881.0,4524.598930,1461.716164,300.0,6800.0,6500.0,...,0.075269,107820.0,846100.0,164243.0,3.72,187,1,train_seg_004,Close,Abnormal resistance
4,484.548387,446.145295,85.0,1993.0,1908.0,4339.247312,1325.237846,300.0,6400.0,6100.0,...,0.070270,90126.0,807100.0,163964.0,3.70,186,1,train_seg_005,Close,Abnormal resistance


In [4]:
# Separate the target and numerical features used for classification.

metadata_columns = ["segment_id", "operation", "status"]
numeric_features = [column for column in cycle_features.columns if column not in metadata_columns]

X = cycle_features[numeric_features].copy()
y = cycle_features["status"].map({"Normal": 0, "Abnormal resistance": 1})

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (110, 140)
y shape: (110,)


In [5]:
# Reproduce the Logistic Regression baseline used in Notebook 4.

RANDOM_STATE = 42
N_SPLITS = 5

cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

baseline = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))])

baseline_pred = np.zeros(len(y), dtype=int)

for train_idx, val_idx in cv.split(X, y):
    baseline.fit(X.iloc[train_idx], y.iloc[train_idx])
    baseline_pred[val_idx] = baseline.predict(X.iloc[val_idx])

print("Accuracy:", accuracy_score(y, baseline_pred))
print("Macro F1:", f1_score(y, baseline_pred, average="macro"))
print("Abnormal recall:", recall_score(y, baseline_pred, zero_division=0))

Accuracy: 1.0
Macro F1: 1.0
Abnormal recall: 1.0


## Chronological Block Validation

The standard validation randomly distributes cycles across folds.

We now preserve their original sequence and evaluate on contiguous
blocks of later cycles.

This tests whether performance depends on random mixing of neighbouring
operations.

In [6]:
# Split the cycles into five chronological validation blocks.

n = len(cycle_features)
fold_size = n // N_SPLITS

chronological_folds = []

for i in range(N_SPLITS):
    start = i * fold_size
    end = n if i == N_SPLITS - 1 else (i + 1) * fold_size
    chronological_folds.append(np.arange(start, end))

for i, fold in enumerate(chronological_folds, start=1):
    print(f"Fold {i}: rows {fold[0]} to {fold[-1]}")

Fold 1: rows 0 to 21
Fold 2: rows 22 to 43
Fold 3: rows 44 to 65
Fold 4: rows 66 to 87
Fold 5: rows 88 to 109


In [7]:
# Evaluate Logistic Regression on each chronological validation block.

chronological_results = []
chronological_predictions = np.zeros(len(y), dtype=int)

for i, val_idx in enumerate(chronological_folds, start=1):
    train_idx = np.setdiff1d(np.arange(len(y)), val_idx)

    model = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))])

    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    pred = model.predict(X.iloc[val_idx])
    chronological_predictions[val_idx] = pred

    chronological_results.append(
        {
            "fold": i,
            "accuracy": accuracy_score(y.iloc[val_idx], pred),
            "balanced_accuracy": balanced_accuracy_score(y.iloc[val_idx], pred),
            "macro_f1": f1_score(y.iloc[val_idx], pred, average="macro"),
            "abnormal_recall": recall_score(y.iloc[val_idx], pred, zero_division=0),
            "abnormal_f1": f1_score(y.iloc[val_idx], pred, zero_division=0),
        }
    )

chronological_results = pd.DataFrame(chronological_results)

display(chronological_results)

,fold,accuracy,balanced_accuracy,macro_f1,abnormal_recall,abnormal_f1
0,1,1.0,1.0,1.0,1.0,1.0
1,2,1.0,1.0,1.0,1.0,1.0
2,3,1.0,1.0,1.0,1.0,1.0
3,4,1.0,1.0,1.0,1.0,1.0
4,5,1.0,1.0,1.0,1.0,1.0


In [8]:
# Summarise the chronological validation performance.

display(chronological_results[["accuracy", "balanced_accuracy", "macro_f1", "abnormal_recall", "abnormal_f1"]].agg(["mean", "std"]))

,accuracy,balanced_accuracy,macro_f1,abnormal_recall,abnormal_f1
mean,1.0,1.0,1.0,1.0,1.0
std,0.0,0.0,0.0,0.0,0.0


In [9]:
# Evaluate several forward-time splits with increasing amounts of training data.

forward_splits = [0.4, 0.5, 0.6, 0.7, 0.8]
forward_results = []

for fraction in forward_splits:
    split = int(len(y) * fraction)

    train_idx = np.arange(0, split)
    val_idx = np.arange(split, len(y))

    if y.iloc[train_idx].nunique() < 2 or y.iloc[val_idx].nunique() < 2:
        continue

    model = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))])

    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    pred = model.predict(X.iloc[val_idx])

    forward_results.append(
        {
            "train_fraction": fraction,
            "train_samples": len(train_idx),
            "validation_samples": len(val_idx),
            "accuracy": accuracy_score(y.iloc[val_idx], pred),
            "balanced_accuracy": balanced_accuracy_score(y.iloc[val_idx], pred),
            "macro_f1": f1_score(y.iloc[val_idx], pred, average="macro"),
            "abnormal_recall": recall_score(y.iloc[val_idx], pred, zero_division=0),
            "abnormal_f1": f1_score(y.iloc[val_idx], pred, zero_division=0),
        }
    )

forward_results = pd.DataFrame(forward_results)

display(forward_results)

,train_fraction,train_samples,validation_samples,accuracy,balanced_accuracy,macro_f1,abnormal_recall,abnormal_f1
0,0.4,44,66,1.0,1.0,1.0,1.0,1.0
1,0.5,55,55,1.0,1.0,1.0,1.0,1.0
2,0.6,66,44,1.0,1.0,1.0,1.0,1.0
3,0.7,77,33,1.0,1.0,1.0,1.0,1.0
4,0.8,88,22,1.0,1.0,1.0,1.0,1.0


In [10]:
# Categorise engineered features according to how they were constructed.


def get_feature_group(feature):
    if "_phase" in feature:
        return "Phase"
    if "_diff_" in feature or feature.startswith("position_abs_diff") or feature.startswith("position_zero_change"):
        return "Dynamic"
    if "integral" in feature:
        return "Integral"
    if feature in ["duration_seconds", "n_rows"]:
        return "Timing"
    if feature == "is_close":
        return "Operation"
    return "Whole-cycle"


feature_groups = pd.DataFrame({"feature": X.columns, "group": [get_feature_group(feature) for feature in X.columns]})

display(feature_groups["group"].value_counts())

group
Phase          99
Whole-cycle    19
Dynamic        16
Integral        3
Timing          2
Operation       1
Name: count, dtype: int64

In [11]:
# Evaluate Logistic Regression using one feature group at a time.

group_results = []

for group in feature_groups["group"].unique():
    selected = feature_groups.loc[feature_groups["group"] == group, "feature"].tolist()

    X_group = X[selected]

    model = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))])

    pred = np.zeros(len(y), dtype=int)

    for train_idx, val_idx in cv.split(X_group, y):
        model.fit(X_group.iloc[train_idx], y.iloc[train_idx])
        pred[val_idx] = model.predict(X_group.iloc[val_idx])

    group_results.append(
        {
            "feature_group": group,
            "feature_count": len(selected),
            "accuracy": accuracy_score(y, pred),
            "macro_f1": f1_score(y, pred, average="macro"),
            "abnormal_recall": recall_score(y, pred, zero_division=0),
            "abnormal_f1": f1_score(y, pred, zero_division=0),
        }
    )

group_results = pd.DataFrame(group_results)

display(group_results.sort_values("accuracy", ascending=False))

,feature_group,feature_count,accuracy,macro_f1,abnormal_recall,abnormal_f1
0,Whole-cycle,19,1.000000,1.000000,1.000000,1.000000
1,Phase,99,0.990909,0.988420,0.966667,0.983051
2,Dynamic,16,0.990909,0.988420,0.966667,0.983051
3,Integral,3,0.990909,0.988420,0.966667,0.983051
4,Timing,2,0.727273,0.421053,0.000000,0.000000
5,Operation,1,0.727273,0.421053,0.000000,0.000000


In [12]:
# Test whether removing an entire feature group harms classification performance.

ablation_results = []

all_groups = feature_groups["group"].unique()

for removed_group in all_groups:
    selected = feature_groups.loc[feature_groups["group"] != removed_group, "feature"].tolist()

    X_ablation = X[selected]

    model = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))])

    pred = np.zeros(len(y), dtype=int)

    for train_idx, val_idx in cv.split(X_ablation, y):
        model.fit(X_ablation.iloc[train_idx], y.iloc[train_idx])
        pred[val_idx] = model.predict(X_ablation.iloc[val_idx])

    ablation_results.append(
        {
            "removed_group": removed_group,
            "remaining_features": len(selected),
            "accuracy": accuracy_score(y, pred),
            "macro_f1": f1_score(y, pred, average="macro"),
            "abnormal_recall": recall_score(y, pred, zero_division=0),
            "abnormal_f1": f1_score(y, pred, zero_division=0),
        }
    )

ablation_results = pd.DataFrame(ablation_results)

display(ablation_results.sort_values("accuracy", ascending=False))

,removed_group,remaining_features,accuracy,macro_f1,abnormal_recall,abnormal_f1
1,Phase,41,1.000000,1.00000,1.000000,1.000000
2,Dynamic,124,1.000000,1.00000,1.000000,1.000000
3,Integral,137,1.000000,1.00000,1.000000,1.000000
4,Timing,138,1.000000,1.00000,1.000000,1.000000
5,Operation,139,1.000000,1.00000,1.000000,1.000000
0,Whole-cycle,121,0.990909,0.98842,0.966667,0.983051


In [22]:
# Evaluate several compact feature sets using SelectKBest inside the CV pipeline.

feature_sizes = [3, 5, 10, 20, 40, "all"]

compact_results = []

for k in feature_sizes:
    selector = SelectKBest(score_func=f_classif, k=k)

    model = Pipeline([("selector", selector), ("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))])

    pred = np.zeros(len(y), dtype=int)

    for train_idx, val_idx in cv.split(X, y):
        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred[val_idx] = model.predict(X.iloc[val_idx])

    compact_results.append(
        {
            "feature_count": k,
            "accuracy": accuracy_score(y, pred),
            "macro_f1": f1_score(y, pred, average="macro"),
            "abnormal_recall": recall_score(y, pred, zero_division=0),
            "abnormal_f1": f1_score(y, pred, zero_division=0),
        }
    )

compact_results = pd.DataFrame(compact_results)

display(compact_results)

,feature_count,accuracy,macro_f1,abnormal_recall,abnormal_f1
0,3,0.990909,0.98842,0.966667,0.983051
1,5,0.990909,0.98842,0.966667,0.983051
2,10,0.990909,0.98842,0.966667,0.983051
3,20,0.990909,0.98842,0.966667,0.983051
4,40,1.000000,1.00000,1.000000,1.000000
5,all,1.000000,1.00000,1.000000,1.000000


In [14]:
# Display the features selected when using the smallest high-performing feature set.

compact_selector = SelectKBest(score_func=f_classif, k=5)
compact_selector.fit(X, y)

compact_features = X.columns[compact_selector.get_support()].tolist()

print("Five selected features:")

for feature in compact_features:
    print(feature)

Five selected features:
Motor current(mA)_phase3_max
Motor current(mA)_phase3_range
Motor Voltage(10mV)_phase2_range
current_abs_integral
voltage_abs_integral


In [15]:
# Evaluate the fixed five-feature set without reselecting features inside validation.

X_five = X[compact_features]

fixed_five_pred = np.zeros(len(y), dtype=int)

for train_idx, val_idx in cv.split(X_five, y):
    model = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))])

    model.fit(X_five.iloc[train_idx], y.iloc[train_idx])
    fixed_five_pred[val_idx] = model.predict(X_five.iloc[val_idx])

print_results = {
    "accuracy": accuracy_score(y, fixed_five_pred),
    "macro_f1": f1_score(y, fixed_five_pred, average="macro"),
    "abnormal_precision": precision_score(y, fixed_five_pred, zero_division=0),
    "abnormal_recall": recall_score(y, fixed_five_pred, zero_division=0),
    "abnormal_f1": f1_score(y, fixed_five_pred, zero_division=0),
}

display(pd.DataFrame([print_results]))

,accuracy,macro_f1,abnormal_precision,abnormal_recall,abnormal_f1
0,0.990909,0.98842,1.0,0.966667,0.983051


In [16]:
# Repeat Logistic Regression validation using several randomised fold assignments.

seeds = [1, 7, 21, 42, 99, 123, 2026]

seed_results = []

for seed in seeds:
    seed_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

    pred = np.zeros(len(y), dtype=int)

    for train_idx, val_idx in seed_cv.split(X, y):
        model = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))])

        model.fit(X.iloc[train_idx], y.iloc[train_idx])
        pred[val_idx] = model.predict(X.iloc[val_idx])

    seed_results.append(
        {
            "seed": seed,
            "accuracy": accuracy_score(y, pred),
            "macro_f1": f1_score(y, pred, average="macro"),
            "abnormal_recall": recall_score(y, pred, zero_division=0),
            "abnormal_f1": f1_score(y, pred, zero_division=0),
        }
    )

seed_results = pd.DataFrame(seed_results)

display(seed_results)

,seed,accuracy,macro_f1,abnormal_recall,abnormal_f1
0,1,1.0,1.0,1.0,1.0
1,7,1.0,1.0,1.0,1.0
2,21,1.0,1.0,1.0,1.0
3,42,1.0,1.0,1.0,1.0
4,99,1.0,1.0,1.0,1.0
5,123,1.0,1.0,1.0,1.0
6,2026,1.0,1.0,1.0,1.0


In [17]:
# Summarise how stable the model is across different CV partitions.

display(seed_results[["accuracy", "macro_f1", "abnormal_recall", "abnormal_f1"]].agg(["mean", "std", "min", "max"]))

,accuracy,macro_f1,abnormal_recall,abnormal_f1
mean,1.0,1.0,1.0,1.0
std,0.0,0.0,0.0,0.0
min,1.0,1.0,1.0,1.0
max,1.0,1.0,1.0,1.0


In [18]:
# Rank individual features by their univariate ANOVA F-score for exploratory inspection.

scores, p_values = f_classif(X, y)

feature_scores = pd.DataFrame({"feature": X.columns, "f_score": scores, "p_value": p_values}).sort_values("f_score", ascending=False)

display(feature_scores.head(30))

,feature,f_score,p_value
135,voltage_abs_integral,421.849747,4.298833e-39
134,current_abs_integral,320.561850,4.188965e-34
32,Motor current(mA)_phase3_max,279.487991,9.830942e-32
33,Motor current(mA)_phase3_range,209.708662,4.652994e-27
53,Motor Voltage(10mV)_phase2_range,201.769235,1.836713e-26
29,Motor current(mA)_phase3_mean,191.473379,1.149971e-25
30,Motor current(mA)_phase3_std,147.152530,6.895671e-22
50,Motor Voltage(10mV)_phase2_std,135.456688,8.841992e-21
126,current_diff_min,92.490240,3.463966e-16
2,Motor current(mA)_min,89.088159,8.814419e-16


In [21]:
# Summarise the robustness experiments performed entirely within this notebook.

print("Randomised stratified CV:")
display(seed_results[["seed", "accuracy", "macro_f1", "abnormal_recall", "abnormal_f1"]])

print("\nChronological block CV:")
display(chronological_results[["fold", "accuracy", "macro_f1", "abnormal_recall", "abnormal_f1"]])

print("\nForward-time validation:")
display(forward_results)

Randomised stratified CV:


,seed,accuracy,macro_f1,abnormal_recall,abnormal_f1
0,1,1.0,1.0,1.0,1.0
1,7,1.0,1.0,1.0,1.0
2,21,1.0,1.0,1.0,1.0
3,42,1.0,1.0,1.0,1.0
4,99,1.0,1.0,1.0,1.0
5,123,1.0,1.0,1.0,1.0
6,2026,1.0,1.0,1.0,1.0



Chronological block CV:


,fold,accuracy,macro_f1,abnormal_recall,abnormal_f1
0,1,1.0,1.0,1.0,1.0
1,2,1.0,1.0,1.0,1.0
2,3,1.0,1.0,1.0,1.0
3,4,1.0,1.0,1.0,1.0
4,5,1.0,1.0,1.0,1.0



Forward-time validation:


,train_fraction,train_samples,validation_samples,accuracy,balanced_accuracy,macro_f1,abnormal_recall,abnormal_f1
0,0.4,44,66,1.0,1.0,1.0,1.0,1.0
1,0.5,55,55,1.0,1.0,1.0,1.0,1.0
2,0.6,66,44,1.0,1.0,1.0,1.0,1.0
3,0.7,77,33,1.0,1.0,1.0,1.0,1.0
4,0.8,88,22,1.0,1.0,1.0,1.0,1.0
